# 03 - Exploratory Data Analysis (EDA)
**Project:** DS Job Recommend  
**Branch:** `feature/visualization`  
**Dataset:** `data/processed/postings_clean.csv`  
**Target:** `formatted_experience_level`

---
### TABLE OF CONTENTS
1. **Environment Setup & Data Loading**
2. **Univariate Analysis** (Numerical & Categorical)
3. **Feature ↔ Feature Analysis** (Correlation & Multicollinearity)
4. **Feature ↔ Target Analysis** (Group Comparison)
5. **Categorical ↔ Numerical Analysis**
6. **Categorical ↔ Categorical Analysis** (Crosstab)
7. **Multivariate Analysis** (Scatter Matrix)
8. **Outlier Analysis & Log Transformation**
9. **Class Imbalance Analysis & Modeling Recommendations**
10. **Skill Analysis**
11. **Text Analysis**
12. **Business Insights & Executive Dashboard**
---

## 1. Environment Setup & Data Loading

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import scatter_matrix
warnings.filterwarnings("ignore")

TARGET = 'formatted_experience_level'

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path('__file__').resolve().parent.parent if '__file__' in locals() else Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

if (ROOT_DIR / 'data' / 'processed').exists():
    PROCESSED = ROOT_DIR / 'data' / 'processed'
elif Path('../data/processed').exists():
    PROCESSED = Path('../data/processed')
else:
    PROCESSED = Path('data/processed')

df = pd.read_csv(PROCESSED / 'postings_clean.csv', low_memory=False)
print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(df.columns.tolist())
print(df.head())
print(df.tail())

In [ ]:
df.info()
df.describe(include='all')

In [ ]:
print('=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== MISSING RATIO (%) ===')
print((df.isnull().sum() / df.shape[0]) * 100)
print(f'\n=== DUPLICATE RATIO: {(df.duplicated().sum() / df.shape[0]) * 100:.2f}% ===')

## 2. Univariate Analysis (Numerical & Categorical)

### 2.1 Numerical Features (Histplot + KDE)

In [ ]:
NUM_COLS = [c for c in ['normalized_salary', 'views', 'applies']
            if c in df.columns]

for col in NUM_COLS:
    data = df[col].dropna()
    
    plt.figure(figsize=(10, 4))
    sns.histplot(data, kde=True)
    plt.title(f'{col} Distribution')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()
    
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    print(f'>>> {col}:')
    print(f'    Spread from {data.min():,.0f} to {data.max():,.0f}')
    print(f'    Concentrated in range {q1:,.0f} - {q3:,.0f} (Q1-Q3)')
    print(f'    Mean: {data.mean():,.0f} | Median: {data.median():,.0f}')
    print()

### 2.2 Categorical Features (Countplot)

In [ ]:
CAT_COLS = [c for c in ['formatted_work_type', 'remote_allowed', 'pay_period']
            if c in df.columns]

for col in CAT_COLS:
    plt.figure(figsize=(10, 4))
    order = df[col].value_counts().index
    sns.countplot(x=col, data=df, order=order, palette='Set2')
    plt.title(f'{col} Distribution')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
    
    counts = df[col].value_counts()
    pct = (counts / len(df) * 100).round(1)
    for val, p in pct.items():
        print(f'    {val}: {p}%')
    print()

## 3. Feature ↔ Feature Analysis (Correlation & Multicollinearity)

In [ ]:
all_num = [c for c in ['min_salary', 'med_salary', 'max_salary',
                        'normalized_salary', 'views', 'applies']
           if c in df.columns]

if len(all_num) >= 2:
    corr = df[all_num].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True)
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.show()
    
    print('>>> Multicollinearity check:')
    for i in range(len(all_num)):
        for j in range(i+1, len(all_num)):
            r = corr.iloc[i, j]
            if abs(r) > 0.9:
                print(f'    WARNING: {all_num[i]} <-> {all_num[j]}: r = {r:.3f} (very high!)')
            elif abs(r) > 0.7:
                print(f'    NOTE: {all_num[i]} <-> {all_num[j]}: r = {r:.3f} (high)')

## 4. Feature ↔ Target Analysis (Group Comparison)

In [ ]:
sal_col = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)

if sal_col and TARGET in df.columns:
    plot_df = df[[TARGET, sal_col]].dropna()
    order = plot_df.groupby(TARGET)[sal_col].median().sort_values(ascending=False).index
    
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=plot_df, x=TARGET, y=sal_col, order=order, palette='Set2')
    plt.title(f'Salary ({sal_col}) by Experience Level')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
if sal_col and TARGET in df.columns:
    avg_salary = df.groupby(TARGET)[sal_col].mean().sort_values(ascending=False)
    
    plt.figure(figsize=(12, 5))
    sns.barplot(x=avg_salary.index, y=avg_salary.values, palette='viridis')
    plt.title(f'Average Salary ({sal_col}) by Experience Level')
    plt.ylabel(f'Average {sal_col}')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
    
    print('>>> Average salary by level:')
    for level, sal in avg_salary.items():
        print(f'    {level}: ${sal:,.0f}')

## 5. Categorical ↔ Numerical Analysis

In [ ]:
if sal_col and 'formatted_work_type' in df.columns:
    plot_df = df[['formatted_work_type', sal_col]].dropna()
    order = plot_df.groupby('formatted_work_type')[sal_col].median().sort_values(ascending=False).index
    
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=plot_df, x='formatted_work_type', y=sal_col, order=order, palette='Set2')
    plt.title(f'Salary by Work Type')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
if sal_col and 'remote_allowed' in df.columns:
    plot_df = df[['remote_allowed', sal_col]].dropna()
    
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=plot_df, x='remote_allowed', y=sal_col, palette='Set2')
    plt.title('Salary by Remote Allowed')
    plt.tight_layout()
    plt.show()

## 6. Categorical ↔ Categorical Analysis (Crosstab)

In [ ]:
if TARGET in df.columns and 'formatted_work_type' in df.columns:
    ct = pd.crosstab(df[TARGET], df['formatted_work_type'], normalize='index') * 100
    print('=== Work Type by Experience Level (%) ===')
    print(ct.round(1))
    
    ct.plot(kind='bar', stacked=True, figsize=(12, 5), colormap='Set2')
    plt.title('Work Type Distribution by Experience Level')
    plt.ylabel('Percentage (%)')
    plt.xticks(rotation=30)
    plt.legend(bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

In [ ]:
if TARGET in df.columns and 'remote_allowed' in df.columns:
    ct2 = pd.crosstab(df[TARGET], df['remote_allowed'], normalize='index') * 100
    print('=== Remote Allowed by Experience Level (%) ===')
    print(ct2.round(1))
    
    ct2.plot(kind='bar', figsize=(10, 5), colormap='Set2')
    plt.title('Remote Allowed by Experience Level')
    plt.ylabel('Percentage (%)')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 7. Multivariate Analysis (Scatter Matrix)

In [ ]:
scatter_cols = [c for c in ['normalized_salary', 'views', 'applies']
                if c in df.columns]

if len(scatter_cols) >= 2:
    sample = df[scatter_cols].dropna()
    if len(sample) > 3000:
        sample = sample.sample(3000, random_state=42)
    scatter_matrix(sample, figsize=(15, 15))
    plt.suptitle('Scatter Matrix - Feature Relationships', fontsize=14)
    plt.tight_layout()
    plt.show()

## 8. Outlier Analysis & Log Transformation

In [ ]:
if NUM_COLS:
    n = len(NUM_COLS)
    ncols = min(n, 3)
    nrows = (n + ncols - 1) // ncols
    
    df[NUM_COLS].plot(kind='box', subplots=True, layout=(nrows, ncols),
                      figsize=(5*ncols, 4*nrows))
    plt.suptitle('Outlier Detection (Boxplot)', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
if sal_col and sal_col in df.columns:
    data = df[sal_col].dropna()
    data_log = np.log1p(data)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    sns.histplot(data, kde=True, ax=axes[0])
    axes[0].set_title(f'Before: {sal_col} (Skewed)')
    
    sns.histplot(data_log, kde=True, ax=axes[1], color='green')
    axes[1].set_title(f'After: log(1 + {sal_col}) (More Normal)')
    
    plt.tight_layout()
    plt.show()
    
    print(f'>>> Skewness before log: {data.skew():.2f}')
    print(f'>>> Skewness after log:  {data_log.skew():.2f}')

## 9. Class Imbalance Analysis & Modeling Recommendations

In [ ]:
if TARGET in df.columns:
    counts = df[TARGET].value_counts()
    print('=== TARGET DISTRIBUTION ===')
    print(counts)
    print(f'\n=== Imbalance Ratio: {counts.max()} / {counts.min()} = {counts.max()/counts.min():.1f} : 1 ===')
    
    plt.figure(figsize=(10, 5))
    sns.countplot(x=TARGET, data=df.dropna(subset=[TARGET]),
                  order=counts.index, palette='Set2')
    plt.title('Experience Level Distribution (Target Variable)')
    plt.ylabel('Count')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
    
    print('\n>>> Modeling Recommendations:')
    print('    1. Use Stratified K-Fold to preserve class ratios')
    print('    2. Use F1-Macro or Balanced Accuracy instead of Accuracy')
    print('    3. Apply SMOTE or class_weight="balanced" to handle imbalance')

## 10. Skill Analysis

In [ ]:
skills_path = PROCESSED / 'job_skills_clean.csv'
if skills_path.exists():
    job_skills = pd.read_csv(skills_path)
    skill_col = next((c for c in ['skill_abr', 'skill_name', 'skill'] if c in job_skills.columns), None)
    
    if skill_col:
        top_skills = job_skills[skill_col].value_counts().head(15)
        
        plt.figure(figsize=(12, 5))
        sns.barplot(x=top_skills.values, y=top_skills.index, palette='viridis')
        plt.title('Top 15 Most Common Skills')
        plt.xlabel('Count')
        plt.tight_layout()
        plt.show()
        
        # Skills by experience level
        id_col = next((c for c in ['job_id', 'jobId'] 
                       if c in job_skills.columns and c in df.columns), None)
        if id_col and TARGET in df.columns:
            merged = job_skills.merge(df[[id_col, TARGET]].dropna(), on=id_col)
            print('\n>>> Top 5 skills by Experience Level:')
            for level in sorted(merged[TARGET].unique()):
                top5 = merged[merged[TARGET] == level][skill_col].value_counts().head(5)
                print(f'\n  {level}:')
                for skill, cnt in top5.items():
                    print(f'    - {skill}: {cnt}')
else:
    print('job_skills_clean.csv not found, skipping skill analysis.')

## 11. Text Analysis

In [ ]:
if 'title' in df.columns:
    df['title_length'] = df['title'].fillna('').str.len()
if 'description' in df.columns:
    df['desc_length'] = df['description'].fillna('').str.len()
    df['desc_word_count'] = df['description'].fillna('').str.split().str.len()

text_cols = [c for c in ['title_length', 'desc_length', 'desc_word_count'] if c in df.columns]

for col in text_cols:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f'{col} Distribution')
    plt.tight_layout()
    plt.show()

# Description length by experience level
if TARGET in df.columns and 'desc_length' in df.columns:
    plot_df = df[[TARGET, 'desc_length']].dropna()
    order = plot_df.groupby(TARGET)['desc_length'].median().sort_values(ascending=False).index
    
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=plot_df, x=TARGET, y='desc_length', order=order, palette='Set2')
    plt.title('Description Length by Experience Level')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 12. Business Insights & Executive Dashboard

In [ ]:
print('=' * 60)
print('       BUSINESS INSIGHTS - DS JOB RECOMMEND')
print('=' * 60)

if TARGET in df.columns:
    counts = df[TARGET].value_counts()
    print(f'\n1. Most posted level: {counts.idxmax()} ({counts.max():,} postings)')
    print(f'   Least posted level: {counts.idxmin()} ({counts.min():,} postings)')

if sal_col and TARGET in df.columns:
    by_level = df.groupby(TARGET)[sal_col].median().sort_values(ascending=False)
    print(f'\n2. Median salary by level:')
    for level, val in by_level.items():
        print(f'   {level}: ${val:,.0f}')

print(f'\n3. Multicollinearity: Salary columns are highly correlated')
print(f'   -> Keep only normalized_salary for modeling')

print(f'\n4. Class imbalance ratio: {counts.max()/counts.min():.0f}:1')
print(f'   -> Must use SMOTE or class_weight')
print('=' * 60)

In [ ]:
if sal_col and TARGET in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # 1. Target Distribution
    target_counts = df[TARGET].dropna().value_counts()
    axes[0, 0].bar([str(x) for x in target_counts.index], target_counts.values, color=sns.color_palette('Set2'))
    axes[0, 0].set_title('1. Experience Level Distribution')
    axes[0, 0].tick_params(axis='x', rotation=30)
    
    # 2. Salary by Level
    order = df.groupby(TARGET)[sal_col].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=TARGET, y=sal_col, order=order, palette='Set2', ax=axes[0, 1])
    axes[0, 1].set_title('2. Salary by Experience Level')
    axes[0, 1].tick_params(axis='x', rotation=30)
    
    # 3. Correlation Heatmap
    sal_cols = [c for c in ['min_salary', 'med_salary', 'max_salary', 'normalized_salary']
                if c in df.columns]
    if len(sal_cols) >= 2:
        sns.heatmap(df[sal_cols].corr(), annot=True, fmt='.2f', ax=axes[1, 0])
        axes[1, 0].set_title('3. Salary Correlation')
    
    # 4. Views vs Applies
    if 'views' in df.columns and 'applies' in df.columns:
        va = df[['views', 'applies']].dropna().sample(min(2000, len(df)), random_state=42)
        axes[1, 1].scatter(va['views'], va['applies'], alpha=0.3, s=10)
        axes[1, 1].set_title('4. Views vs Applies')
        axes[1, 1].set_xlabel('Views')
        axes[1, 1].set_ylabel('Applies')
    
    plt.suptitle('EXECUTIVE DASHBOARD', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('\nExploratory Data Analysis completed!')